## Importing modules needed

In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

#### Loading data

In [2]:
df20 = pd.read_csv('churn-bigml-20.csv')
df80 = pd.read_csv('churn-bigml-80.csv')

In [4]:
print(f'Shape of df80 is {df80.shape}')  

Shape of df80 is (2666, 20)


In [5]:
print(f' shape of df20 is{df20.shape}')

 shape of df20 is(667, 20)


In [ ]:
df = pd.concat([df20, df80],ignore_index= True)  # Concatenate the two DataFrames

In [8]:
print(df.shape) # Check the shape of the combined DataFrame

(3333, 20)


In [9]:
print(df.isnull().sum())  # Check for missing values

State                     0
Account length            0
Area code                 0
International plan        0
Voice mail plan           0
Number vmail messages     0
Total day minutes         0
Total day calls           0
Total day charge          0
Total eve minutes         0
Total eve calls           0
Total eve charge          0
Total night minutes       0
Total night calls         0
Total night charge        0
Total intl minutes        0
Total intl calls          0
Total intl charge         0
Customer service calls    0
Churn                     0
dtype: int64


In [10]:
# Imputing numeric columns with median
num_col = df.select_dtypes(include="number").columns.tolist()

In [11]:
imputer = SimpleImputer(strategy='median') # Create an imputer object with the desired strategy

In [12]:
df[num_col] = imputer.fit_transform(df[num_col]) # Apply the imputation to the numerical columns

In [13]:
print(df.isnull().sum().sum())  #missing values remaing

0


In [14]:
#remove outliers using IQR(Inter Quatile Range) method
before = df.shape[0]

In [ ]:
for col in num_col:
    """    Calculate the first quartile (Q1) and third quartile (Q3) for the column."""
    Q1 = df[col].quantile(0.75)  
    Q3 = df[col].quantile(0.25)
    IQR = Q3 - Q1 # Calculate the interquartile range (IQR)
    df = df[((df[col] <Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR))] # Remove outliers based on the IQR method
df = df.reset_index(drop=True)

In [ ]:
print(f' Rows before:{before}, after:{df.shape[0]}, removed:{before - df.shape[0]}') # Print the number of rows before and after outlier removal, and the number of rows removed

 Rows before:3333, after:3333, removed:0


In [29]:
#Encoding categorical variables
le = LabelEncoder()  #encoding for ordinal columns

In [ ]:
for col in ['International plan','Voice mail plan']: # Loop through the specified columns and apply label encoding
    df[col] = le.fit_transform(df[col].astype(str))

In [ ]:
df = pd.get_dummies(df, columns=['State'], drop_first =True) # Apply one-hot encoding to the 'State' column, dropping the first category to avoid multicollinearity

In [ ]:
df['Churn'] = df['Churn'].astype(int)  # Convert the 'Churn' column to integer type

In [ ]:
print(f'Shape: {df.shape}')
df.dtypes.value_counts() # Check the data types of the columns and count the occurrences of each data type

Shape: (3333, 69)


bool       50
float64    16
int64       3
Name: count, dtype: int64

In [ ]:
# Standardiza numerical data
feature_col = [c for c in df.select_dtypes(include='number').columns if c != 'Churn'] # Select all numerical columns except the target variable 'Churn'
scaler = StandardScaler() # Create a StandardScaler object to standardize the numerical features by removing the mean and scaling to unit variance
df[feature_col] = scaler.fit_transform(df[feature_col]) # Apply the standardization to the selected numerical columns

In [ ]:
print(df[feature_col[:5]].describe().round(3)) # Print the descriptive statistics of the first five standardized numerical features, rounded to three decimal places

       Account length  Area code  International plan  Voice mail plan  \
count        3333.000   3333.000            3333.000         3333.000   
mean            0.000      0.000              -0.000           -0.000   
std             1.000      1.000               1.000            1.000   
min            -2.513     -0.689              -0.328           -0.618   
25%            -0.680     -0.689              -0.328           -0.618   
50%            -0.002     -0.524              -0.328           -0.618   
75%             0.651      1.719              -0.328            1.617   
max             3.565      1.719               3.053            1.617   

       Number vmail messages  
count               3333.000  
mean                   0.000  
std                    1.000  
min                   -0.592  
25%                   -0.592  
50%                   -0.592  
75%                    0.870  
max                    3.135  


In [ ]:
df.shape # Check the shape of the cleaned DataFrame

(3333, 69)

In [ ]:
df.to_csv('churn_cleaned_csv', index=False) # Save the cleaned DataFrame to a new CSV file without the index